In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')
LOG_DIR = "/content/drive/MyDrive/Research_Logs"

print(f"✅ ログ保存先: {LOG_DIR}")

%pip install rank_bm25
%pip install xopen

from pathlib import Path
import sys
import torch
from datasets import load_dataset
from datetime import datetime
import xopen
import random

import huggingface_hub
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
huggingface_hub.login(hf_token)

random.seed(0)


# 以下自作モジュール
module_path = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/my_modules"
if module_path not in sys.path:
    sys.path.append(module_path)

if 'CACHED_MODEL' not in globals():
    CACHED_MODEL = None
    CACHED_TOKENIZER = None

def confirmation():
    """実行前にユーザーの確認を求める"""
    print("\n" + "!" * 30)
    print("警告: 書き出しモードが 'a' (追記) です。")
    print("既存のファイルにデータが追加されますが、よろしいですか？")
    print("!" * 30 + "\n")

    answer = input("実行する場合は 'yes' と入力してください: ").strip().lower()

    if answer != 'yes':
        print("\n[中止] 実行がキャンセルされました。")
        # Jupyterでセルを止める最もクリーンな方法
        raise KeyboardInterrupt("User cancelled the execution.")

    print("\n[開始] 実験を実行します...\n")

Mounted at /content/drive
✅ ログ保存先: /content/drive/MyDrive/Research_Logs
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.4/132.4 kB 2.3 MB/s eta 0:00:00


In [2]:
import torch
import sys
from transformers import AutoTokenizer, AutoModelForCausalLM, LlamaTokenizer
from DynamicScalingLlamaAttention_last_attnweight import DynamicScalingLlamaAttention
CACHED_MODEL = None
CACHED_TOKENIZER = None

def get_model_safe(model_path):
    global CACHED_MODEL, CACHED_TOKENIZER

    if CACHED_MODEL is not None:
        print("✅ Model already loaded. Using cached model.")
        return CACHED_MODEL, CACHED_TOKENIZER

    print("⏳ Loading new model... (This may take a while)")

    tokenizer = LlamaTokenizer.from_pretrained(model_path, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map="auto",
        attn_implementation="eager",
        dtype=torch.float16,
        use_cache=False,
        trust_remote_code=True
    )

    if len(model.model.layers) > 20:
        print(f"✂️ Truncating model layers from {len(model.model.layers)} to 20")
        model.model.layers = model.model.layers[:20]

    gc.collect()
    torch.cuda.empty_cache()

    print("🔄 Swapping Attention Modules to DynamicScalingLlamaAttention...")

    target_layers = list(range(14,20))

    # モデル設定
    model.config.hidden_scale_config = {
        "target_layers": target_layers,
        "target_dims": [2393],
        "factor": 1,
        "last_recompute_tokens": 1,
        "change_value": False
    }

    # 全レイヤー入れ替え
    for i, layer in enumerate(model.model.layers):
        if i in target_layers:
            original_attn = layer.self_attn
            target_device = original_attn.q_proj.weight.device
            target_dtype = original_attn.q_proj.weight.dtype

            new_attn = DynamicScalingLlamaAttention(config=model.config, layer_idx=i)

            # originalの情報をコピー
            new_attn.load_state_dict(original_attn.state_dict(), strict=False)
            new_attn.to(device=target_device, dtype=target_dtype)
            layer.self_attn = new_attn
        else: pass

    CACHED_MODEL = model
    CACHED_TOKENIZER = tokenizer

    print("✅ Model loaded and patched successfully.")
    return CACHED_MODEL, CACHED_TOKENIZER

In [3]:
from datasets.formatting.formatting import TypeVar
from typing import List, Optional, Type
from pydantic.dataclasses import dataclass
from rank_bm25 import BM25Okapi
import numpy as np

PROMPTS_ROOT = Path("/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/lost_in_the_middle/prompts")
T = TypeVar("T")

@dataclass(frozen=True)
class Document:
    title: str
    text: str
    id: Optional[str] = None
    score: Optional[float] = None
    hasanswer: Optional[bool] = None
    isgold: Optional[bool] = None
    original_retrieval_index: Optional[int] = None

    @classmethod
    def from_dict(cls: Type[T], data: dict) -> T:
        data = deepcopy(data)
        if not data:
            raise ValueError("Must provide data for creation of Document from dict.")
        id = data.pop("id", None)
        score = data.pop("score", None)
        isgold = data.pop("isgold", None)
        # Convert score to float if it's provided.
        if score is not None:
            score = float(score)
        return cls(**dict(data, id=id, score=score, isgold=isgold))

def get_qa_prompt_reorder(question, documents, trial_orders):
  prompt_filename = "qa.prompt"

  with open(PROMPTS_ROOT / prompt_filename) as f:
    prompt_template = f.read().rstrip("\n")

  all_prompts = []
  all_doc_positions = []

  for trial_num in trial_orders:

    shifted_documents = documents[trial_num:] + documents[:trial_num]

    formatted_documents = []
    for document_index, document in enumerate(shifted_documents):
      formatted_documents.append(f"Document [{document_index+1}](Title: {document.title}) {document.text}")
    prompt = prompt_template.format(question=question, search_results="\n".join(formatted_documents))

    doc_positions = []
    for formatted_document in formatted_documents:
      start_idx = prompt.find(formatted_document)
      end_idx = start_idx + len(formatted_document)
      doc_positions.append({
          "start": start_idx,
          "end": end_idx
          })

    all_prompts.append(prompt)
    all_doc_positions.append(doc_positions)

  return all_prompts, all_doc_positions

def get_attnmap(question, documents, model, tokenizer, trial_orders):
  """各文書のトークンに対するアテンション重みの取得を行う"""
  # プロンプトの作成と位置取得
  all_prompts, all_doc_positions = get_qa_prompt_reorder(question, documents, trial_orders)

  # トークナイズ
  inputs = tokenizer(
      all_prompts,
      return_tensors="pt",
      return_offsets_mapping=True,
  ).to(model.device)
  offset_mapping = inputs.pop("offset_mapping")

  # Hookの設定
  layer_attns = {}
  target_layer_indices = range(14, 20) # 15層(index 14)から20層(index 19)
  def hook_fn(layer_idx):
    def fn(module, input, output):
      attn_weights = module.attn_weights_last
      layer_attns[layer_idx] = attn_weights.detach().cpu()
      module.attn_weights_last = None
    return fn

  hooks = []
  for i in target_layer_indices:
    h = model.model.layers[i].self_attn.register_forward_hook(hook_fn(i))
    hooks.append(h)

  # Prefill
  try:
    with torch.inference_mode():
      _ = model(**inputs)

    # 各文書の重みを集計
    all_selected_attn = torch.stack(list(layer_attns.values())).transpose(0,1) # [batch, layers, heads, 1, seq_len]
    avg_attn_matrix = all_selected_attn.mean(dim=3) # [batch, layers, heads, seq_len]

    # バッチごとに文書のアテンション重みの取得
    batch_doc_weights = []
    for batch_count in range(len(avg_attn_matrix)):
      offsets = torch.tensor(offset_mapping[batch_count])
      s, e = offsets[:, 0], offsets[:, 1]

      # 各文書のアテンションの取得
      current_doc_weights = []
      for pos in all_doc_positions[batch_count]:
        start_char, end_char = pos["start"], pos["end"]

        mask = (s != e) & (e >= start_char) & (s <= end_char)
        doc_token_indices = torch.where(mask)[0].tolist()

        doc_attn_weights = avg_attn_matrix[batch_count, :, :, doc_token_indices] # (layers, heads, doc_len)
        doc_attn = doc_attn_weights.mean(dim=2) # (layers, heads)
        current_doc_weights.append(doc_attn)

      # 並び替え
      num_docs = len(documents)
      trial_order = trial_orders[batch_count]

      original_ordered_scores = [None] * num_docs
      for j, score in enumerate(current_doc_weights):
        original_idx = (j + trial_order) % num_docs
        original_ordered_scores[original_idx] = score

      batch_doc_weights.append(original_ordered_scores)

  finally:
    for h in hooks:
      h.remove()

  return batch_doc_weights


def get_bm25_indices(question, documents):
    """
    質問と文書群を受け取り、BM25スコアが高い順のインデックスリストを返す
    """
    # トークナイズ（簡易版）
    tokenized_corpus = [doc.text.lower().split() for doc in documents]
    bm25 = BM25Okapi(tokenized_corpus)

    tokenized_query = question.lower().split()
    # 文書ごとのスコアを取得
    doc_scores = bm25.get_scores(tokenized_query)

    # スコア降順のインデックスを返す
    # argsortは昇順なので [::-1] で反転させる
    return np.argsort(doc_scores)[::-1].tolist()

In [4]:
from xopen import xopen
from copy import deepcopy
from tqdm import tqdm
import json
import random
import os
import gc
import torch
import subprocess
import time

# --- 設定 ---
SYNC_INTERVAL = 100  # 100件ごとにDriveへ同期
class SafeDriveSaver:
    def __init__(self, target_drive_path: str):
        self.drive_path = target_drive_path
        # ローカルの一時ファイルパス (/content/ファイル名)
        self.local_path = os.path.join("/content", os.path.basename(target_drive_path))

        # 保存先ディレクトリ作成
        os.makedirs(os.path.dirname(self.drive_path), exist_ok=True)

        # 1. 初期ロード (Drive -> Local)
        if os.path.exists(self.drive_path):
            print(f"🔄 Driveから既存データをロード中: {self.drive_path}")
            # エラーチェック付きでコピー
            res = subprocess.run(f'cp "{self.drive_path}" "{self.local_path}"', shell=True)
            if res.returncode != 0:
                raise RuntimeError("❌ Driveからのデータ読み込みに失敗しました。停止します。")
        else:
            print(f"🆕 新規ファイルとして開始します")

    @property
    def temp_file_path(self) -> str:
        return self.local_path

    def get_current_count(self) -> int:
        """現在ローカルに保存されている行数を返す"""
        if not os.path.exists(self.local_path):
            return 0
        with open(self.local_path, "r", encoding="utf-8") as f:
            return sum(1 for _ in f)

    def save_to_local(self, file_handle, data_dict):
        """ローカル一時保存 (高速)"""
        file_handle.write(json.dumps(data_dict) + "\n")
        file_handle.flush()
        os.fsync(file_handle.fileno())

    def sync_to_drive(self):
        """Driveへ同期 (安全版: .bak作成 -> 上書き -> sync)"""
        if not os.path.exists(self.local_path):
            return

        # バックアップ作成
        if os.path.exists(self.drive_path):
            subprocess.run(f'cp -f "{self.drive_path}" "{self.drive_path}.bak"', shell=True)

        # メインの上書き同期
        subprocess.run(f'cp -f "{self.local_path}" "{self.drive_path}"', shell=True)
        subprocess.run('sync', shell=True)

def main():
    # 1. 保存管理クラスの初期化
    # (Driveからローカルへロードし、既存の行数をカウントする)
    saver = SafeDriveSaver(OUTPUT_PATH)

    # 2. 再開位置の特定
    done_count = saver.get_current_count()
    print(f"Resume check: {done_count} lines already processed.")

    # モデルの準備
    model, tokenizer = get_model_safe(MODEL_NAME)
    model.config.current_scale_map = None
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    # 3. ファイルオープン (saver.temp_file_path = ローカルパス)
    with xopen(NQ_path, "r") as fin, open(saver.temp_file_path, "a", encoding="utf-8") as fout:

        # tqdmの設定：全処理対象数
        total_to_process = DATA_NUM - VALIDATION_SIZE
        pbar = tqdm(total=total_to_process, desc="Processing")

        # 既読分だけバーを更新
        if done_count > 0:
            pbar.update(done_count)

        processed_in_session = 0

        for i, line in enumerate(fin):
            # スキップロジック
            if i < VALIDATION_SIZE + done_count:
                continue
            if i >= DATA_NUM:
                break

            data = json.loads(line)
            question = data["question"]

            documents = [Document.from_dict(ctx) for ctx in deepcopy(data["ctxs"])]
            if not documents:
                raise ValueError(f"Did not find any documents: {data}")

            # BM25インデックス取得と並べ替え
            bm25_indices = get_bm25_indices(question, documents)
            bm25_reordered_documents = [documents[idx] for idx in bm25_indices]

            all_attnmaps = []

            # バッチ処理ループ
            for start_idx in range(0, SHUFFLE_NUM, BATCH_SIZE):
                end_idx = min(start_idx + BATCH_SIZE, SHUFFLE_NUM)
                trial_orders = list(range(start_idx, end_idx))

                attnmaps = get_attnmap(
                    question,
                    bm25_reordered_documents,
                    model, tokenizer, trial_orders
                )

                for trial_attnmap in attnmaps:
                    restored_attnmap = [None] * len(documents)
                    for j, score in enumerate(trial_attnmap):
                        original_idx = bm25_indices[j]
                        restored_attnmap[original_idx] = score.tolist()
                    all_attnmaps.append(restored_attnmap)

                del attnmaps
                torch.cuda.empty_cache()

            # 出力データ作成
            output_dict = {
                "id": i,
                "question": question,
                "attnmap": all_attnmaps,
            }

            # --- 保存処理 ---
            # 1. ローカルへ即時書き込み
            saver.save_to_local(fout, output_dict)

            processed_in_session += 1
            pbar.update(1)

            # 2. 定期的にDriveへ同期
            if processed_in_session % SYNC_INTERVAL == 0:
                pbar.set_description("Syncing to Drive...")
                saver.sync_to_drive()
                pbar.set_description("Processing")

    # 4. 最終同期
    print("Finalizing save and syncing to Drive...")
    saver.sync_to_drive()

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print(f"✅ Processing complete. Output: {OUTPUT_PATH}")

In [ ]:
NQ_path = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/NQ/qa_data/20_total_documents/nq-open-20_total_documents_gold_at_0.jsonl.gz"
OUTPUT_PATH = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_attn_maps/attnmap_shift_bm25_llama.jsonl"

SHUFFLE_NUM = 20
BATCH_SIZE = 8

VALIDATION_SIZE = 600
DATA_NUM = 2655

MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"

if __name__ == "__main__":
  main()

⏳ Loading new model... (This may take a while)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

🔄 Swapping Attention Modules to DynamicScalingLlamaAttention...
✅ Model loaded and patched successfully.


1464it [00:01, 1270.08it/s]/tmp/ipython-input-643411022.py:105: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  offsets = torch.tensor(offset_mapping[batch_count])
2655it [3:48:34,  5.17s/it]


In [5]:
NQ_path = f"/content/drive/MyDrive/Colab_Notebooks/graduate_research/PositionalHidden/experiments/NQ/qa_data/20_total_documents/nq-open-20_total_documents_gold_at_0.jsonl.gz"
OUTPUT_PATH = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_attn_maps/attnmap_shift_bm25_vicuna-16k.jsonl"

SHUFFLE_NUM = 20
BATCH_SIZE = 10

VALIDATION_SIZE = 600
DATA_NUM = 2655

MODEL_NAME = "lmsys/vicuna-7b-v1.5-16k"

if __name__ == "__main__":
  main()

🔄 Driveから既存データをロード中: /content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_attn_maps/attnmap_shift_bm25_vicuna-16k.jsonl
Resume check: 1380 lines already processed.
⏳ Loading new model... (This may take a while)


tokenizer_config.json:   0%|          | 0.00/750 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/692 [00:00<?, ?B/s]

...

✂️ Truncating model layers from 32 to 20
🔄 Swapping Attention Modules to DynamicScalingLlamaAttention...
✅ Model loaded and patched successfully.


Processing:   0%|          | 0/2055 [00:00<?, ?it/s]/tmp/ipython-input-643411022.py:105: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  offsets = torch.tensor(offset_mapping[batch_count])
Processing: 100%|██████████| 2055/2055 [2:13:18<00:00, 11.30s/it]

Finalizing save and syncing to Drive...


Processing: 100%|██████████| 2055/2055 [2:13:36<00:00,  3.90s/it]

✅ Processing complete. Output: /content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_attn_maps/attnmap_shift_bm25_vicuna-16k.jsonl


In [ ]:
from tqdm import tqdm
path = "/content/drive/MyDrive/Colab_Notebooks/graduate_research/Logs/NQ/validation_attn_maps/attnmap_shift_bm25_vicuna-16k.jsonl"
count = 0
with open(path, "r") as f:
  for i in tqdm(f):
    continue

1943it [00:23, 72.17it/s]

In [ ]:
import time
import gc
import torch
from google.colab import runtime

# 1. メモリの最終開放（念のため）
gc.collect()
torch.cuda.empty_cache()

# 2. 待機設定
# Google Driveの同期には30〜60秒ほど見ておくと非常に安全です
WAIT_TIME_SEC = 300

print(f"✅ 全ての処理が完了しました。")
print(f"📂 Google Driveへの反映を待機しています（{WAIT_TIME_SEC}秒）...")

# プログレスバー付きで待機（あとどれくらいで消えるか視覚化）
from tqdm import tqdm
for _ in tqdm(range(WAIT_TIME_SEC)):
    time.sleep(1)

print("\n🚀 待機完了。セッションを終了し、ランタイムを削除します。お疲れ様でした！")

# 3. セッション終了
runtime.unassign()